# Open-Domain Claim Verification — Tweet Extension Pipeline

Extension of [Vladika & Matthes (EACL 2024)](https://arxiv.org/abs/2402.02844) to a new bilingual dataset of **60 social media posts** (30 English, 30 Spanish) annotated with **TRUE / FALSE / UNVERIFIABLE** labels for scientific claims.

This notebook reuses the pipeline from [`reproduction/api-pipeline/open_domain_claim_verification_api.ipynb`](../reproduction/api-pipeline/open_domain_claim_verification_api.ipynb) (retrieval -> SPICED evidence selection -> DeBERTa-v3-large NLI verdict) **unchanged**, replacing the four benchmark datasets with the tweet dataset.

| Step | Source |
|---|---|
| 1 — Document retrieval | Wikipedia API (`wikipedia` package) + NCBI PubMed E-utilities (`Bio.Entrez`) |
| 2 — Evidence selection | SPICED sentence similarity (unchanged) |
| 3 — Verdict prediction | DeBERTa-v3-large NLI (unchanged) |

**Evaluation:** F1, precision, recall, and accuracy are computed on **TRUE/FALSE posts only** (19/30 EN, 22/30 ES) — UNVERIFIABLE posts have no SUPPORTED/REFUTED ground truth, consistent with how NEI/NOT-ENOUGH-INFO claims are excluded for SciFact/PubMedQA/HealthFC/CoVERT in `reproduction/`. The pipeline still produces a prediction for **every** post (including UNVERIFIABLE ones) so these can be reviewed for the failure taxonomy.

**Estimated runtime on free Google Colab (T4 GPU):** the dataset is tiny (60 claims total) compared to the benchmark runs (up to ~2,300 claims), so retrieval + SPICED + NLI together should take a few minutes per knowledge source. Most of the wall-clock time is loading the SPICED and DeBERTa-v3-large models (~2-5 min).

In [21]:
# Installs packages not pre-installed on Colab.
# torch, numpy, pandas, scikit-learn, nltk are already on Colab — excluded to avoid
# overwriting Colab's GPU-enabled PyTorch with a CPU-only version.
# If running locally, install all packages manually:
#!pip install wikipedia sentence-transformers transformers torch nltk scikit-learn pandas numpy biopython python-dotenv openpyxl
!pip install wikipedia sentence-transformers transformers biopython python-dotenv openpyxl --quiet

## Setup

**Running on Google Colab (VSCode extension or browser):**
1. Run the Drive mount cell below
2. Make sure your Drive has the folder `My Drive/NLP-Group-Project/` with the same structure as the GitHub repo (in particular `data/english/`, `data/spanish/`, and `reproduction/api-pipeline/results/` for the baseline comparison)
3. Upload `.env` to `My Drive/NLP-Group-Project/.env` for PubMed API credentials (see `.env.example` at the project root)

**Running locally:**
1. Skip the Drive mount cell

In [22]:
# Auto-detects whether running on Google Colab and mounts Drive if so.
# No changes needed — works the same locally and on Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Running on Colab — Drive mounted.')
except ImportError:
    print('Running locally — skipping Drive mount.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running on Colab — Drive mounted.


In [23]:
import os
import re
import csv
import time
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import torch
import wikipedia
import nltk
from Bio import Entrez
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from dotenv import load_dotenv

# Wikimedia rate-limits the 'wikipedia' package's default User-Agent
# ('wikipedia (https://github.com/goldsmith/Wikipedia/)') with HTTP 429,
# since it's shared by every user of this unmaintained package. A unique,
# descriptive User-Agent (per https://meta.wikimedia.org/wiki/User-Agent_policy)
# avoids this and lets wikipedia.search()/page() return real results.
wikipedia.wikipedia.USER_AGENT = (
    'NLP-Group-Project/1.0 (https://github.com/am-claudia/NLP-Group-Project; '
    'andrea.alarcon@student.ie.edu) research-extension'
)

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

try:
    import google.colab
    RUNNING_ON_COLAB = True
except ImportError:
    RUNNING_ON_COLAB = False

if RUNNING_ON_COLAB:
    BASE_PATH = '/content/drive/MyDrive/NLP-Group-Project'
else:
    BASE_PATH = '../..'

DATASETS_PATH       = os.path.join(BASE_PATH, 'data')
BASELINE_PATH       = os.path.join(BASE_PATH, 'reproduction', 'api-pipeline', 'results')
WIKI_RESULTS_PATH   = os.path.join(BASE_PATH, 'pipeline', 'results', 'Wikipedia API')
PUBMED_RESULTS_PATH = os.path.join(BASE_PATH, 'pipeline', 'results', 'PubMed API')
COMPARISON_PATH     = os.path.join(BASE_PATH, 'pipeline', 'results', 'comparison')
os.makedirs(WIKI_RESULTS_PATH,   exist_ok=True)
os.makedirs(PUBMED_RESULTS_PATH, exist_ok=True)
os.makedirs(COMPARISON_PATH,     exist_ok=True)

DATASETS_TO_RUN = ['tweets_en', 'tweets_es']

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on Colab: {RUNNING_ON_COLAB}')
print(f'Using device:     {device}')
print(f'Datasets path:    {os.path.abspath(DATASETS_PATH)}')
print(f'Baseline path:    {os.path.abspath(BASELINE_PATH)}')
print(f'Wikipedia results:{os.path.abspath(WIKI_RESULTS_PATH)}')
print(f'PubMed results:   {os.path.abspath(PUBMED_RESULTS_PATH)}')

load_dotenv(os.path.join(BASE_PATH, '.env'), override=True)

Running on Colab: True
Using device:     cuda
Datasets path:    /content/drive/MyDrive/NLP-Group-Project/data
Baseline path:    /content/drive/MyDrive/NLP-Group-Project/reproduction/api-pipeline/results
Wikipedia results:/content/drive/MyDrive/NLP-Group-Project/pipeline/results/Wikipedia API
PubMed results:   /content/drive/MyDrive/NLP-Group-Project/pipeline/results/PubMed API


False

## Functions & Model Setup

All helper functions are defined and models loaded here before any data is processed.

Retrieval, evidence selection, and verdict prediction functions are copied unchanged from `reproduction/api-pipeline/open_domain_claim_verification_api.ipynb`. Two additions for this notebook:
- `compute_metrics()` — scores only the TRUE/FALSE subset (gold label not `None`), but is given predictions for every post
- `save_predictions()` / the failure-table cells later — keep a per-post record (claim, gold, prediction, evidence) for the failure taxonomy

In [24]:
# ── Wikipedia retrieval ───────────────────────────────────────────────────────

def retrieve_wikipedia_articles(claim, n_results=10):
    """
    Search Wikipedia for a claim and return:
      - fetched_titles: list of article titles
      - articles:       list of article text content
    Silently skips disambiguation pages and pages that cannot be fetched.
    """
    try:
        titles = wikipedia.search(claim, results=n_results)
    except Exception:
        return [], []

    fetched_titles = []
    articles = []
    for title in titles:
        try:
            page = wikipedia.page(title, auto_suggest=False)
            fetched_titles.append(page.title)
            articles.append(page.content)
        except (wikipedia.exceptions.DisambiguationError,
                wikipedia.exceptions.PageError,
                Exception):
            continue
        time.sleep(0.3)
    return fetched_titles, articles


# ── PubMed retrieval ──────────────────────────────────────────────────────────

def retrieve_pubmed_articles(claim, n_results=10):
    """
    Search PubMed for a claim via NCBI E-utilities and return:
      - fetched_pmids: list of PubMed IDs
      - articles:      list of 'title + abstract' strings
    Mirrors retrieve_wikipedia_articles() in structure.
    """
    try:
        search_handle = Entrez.esearch(db='pubmed', term=claim, retmax=n_results)
        search_record = Entrez.read(search_handle)
        search_handle.close()
        pmids = search_record['IdList']

        if not pmids:
            return [], []

        time.sleep(0.1 if ncbi_api_key else 0.34)  # 10 req/sec with key, 3 req/sec without

        fetch_handle = Entrez.efetch(db='pubmed', id=pmids, rettype='xml', retmode='xml')
        fetch_data   = fetch_handle.read()
        fetch_handle.close()

        root = ET.fromstring(fetch_data)
        fetched_pmids = []
        articles      = []

        for article in root.findall('.//PubmedArticle'):
            pmid_el  = article.find('.//PMID')
            pmid     = pmid_el.text if pmid_el is not None else ''

            title_el = article.find('.//ArticleTitle')
            title    = (title_el.text or '') if title_el is not None else ''

            abstract_els = article.findall('.//AbstractText')
            abstract = ' '.join(el.text or '' for el in abstract_els if el.text)

            if abstract:
                fetched_pmids.append(pmid)
                articles.append(f'{title} {abstract}')

        return fetched_pmids, articles

    except Exception as e:
        print(f'Error: {e}')
        return [], []


# ── Evidence selection ────────────────────────────────────────────────────────

def select_top_sentences(claim, articles, model, top_k=10):
    """
    Extract all sentences from retrieved articles, then select the top-k
    most similar to the claim using cosine similarity.
    Returns a single string of the selected sentences.
    """
    all_sentences = []
    for article in articles:
        sentences = sent_tokenize(article)
        all_sentences.extend([s.lower() for s in sentences])

    if not all_sentences:
        return ''

    top_k = min(top_k, len(all_sentences))
    sents_embeddings = model.encode(all_sentences, convert_to_tensor=True)
    claim_embedding  = model.encode(claim, convert_to_tensor=True)
    cos_scores = util.cos_sim(claim_embedding, sents_embeddings)[0]

    top_indices = torch.topk(cos_scores, k=top_k).indices.cpu().numpy()
    top_indices = np.sort(top_indices)
    selected = np.array(all_sentences)[top_indices]
    return ' '.join(selected)


# ── Verdict prediction & metrics ──────────────────────────────────────────────

class ClaimDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __getitem__(self, idx):
        item = {k: v[idx].clone().detach() for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


def predict_verdicts(joint_list, model, tokenizer, batch_size=16):
    encodings = tokenizer(
        joint_list, return_tensors='pt', truncation='only_first',
        padding=True, max_length=1024
    )
    dataset = ClaimDataset(encodings, [0] * len(joint_list))
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    model.eval()
    model.to(device)
    predictions = []
    with torch.no_grad():
        for batch in loader:
            logits = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device)
            )[0]
            probs = logits.softmax(dim=1)
            predictions.extend((probs[:, 0] > probs[:, 2]).cpu().numpy().astype(int).tolist())
    return np.array(predictions)


def compute_metrics(gold_labels, predictions):
    """
    Precision/recall/F1/accuracy on the subset of (gold, pred) pairs where
    gold is not None. UNVERIFIABLE posts (gold=None) are excluded from scoring
    but `predictions` still contains a value for them (used in the failure
    taxonomy below).
    """
    pairs = [(g, p) for g, p in zip(gold_labels, predictions) if g is not None]
    n_total = len(gold_labels)
    if not pairs:
        return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.0,
                'n_scored': 0, 'n_total': n_total}
    gold, pred = zip(*pairs)
    return {
        'precision': precision_score(gold, pred, average='binary', zero_division=0),
        'recall':    recall_score(gold, pred, average='binary', zero_division=0),
        'f1':        f1_score(gold, pred, average='binary', zero_division=0),
        'accuracy':  accuracy_score(gold, pred),
        'n_scored':  len(pairs),
        'n_total':   n_total,
    }


def save_metrics(results, save_path):
    fieldnames = ['dataset', 'precision', 'recall', 'f1', 'accuracy', 'n_scored', 'n_total']
    with open(save_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for ds, s in results.items():
            writer.writerow({
                'dataset':   ds,
                'precision': round(s['precision'] * 100, 1),
                'recall':    round(s['recall']    * 100, 1),
                'f1':        round(s['f1']        * 100, 1),
                'accuracy':  round(s['accuracy']  * 100, 1),
                'n_scored':  s['n_scored'],
                'n_total':   s['n_total'],
            })
    print(f'Saved: {save_path}')


def load_metrics(path):
    results = {}
    with open(path, 'r', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            results[row['dataset']] = {
                'precision': float(row['precision']) / 100,
                'recall':    float(row['recall'])    / 100,
                'f1':        float(row['f1'])        / 100,
                'accuracy':  float(row['accuracy'])  / 100,
                'n_scored':  int(row['n_scored']),
                'n_total':   int(row['n_total']),
            }
    return results


LABEL_NAMES = {1: 'SUPPORTED', 0: 'REFUTED', None: 'UNVERIFIABLE'}


def save_predictions(ids, claims, gold_labels, predictions, joint_list, save_path):
    """Per-post predictions for the failure taxonomy — includes UNVERIFIABLE posts."""
    with open(save_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=[
            'id', 'claim', 'gold_label', 'predicted_label', 'correct', 'evidence'
        ])
        writer.writeheader()
        for tid, claim, gold, pred, joint in zip(ids, claims, gold_labels, predictions, joint_list):
            evidence = joint.split('[SEP]', 1)[1].strip() if '[SEP]' in joint else ''
            correct = '' if gold is None else ('Y' if gold == pred else 'N')
            writer.writerow({
                'id': tid,
                'claim': claim,
                'gold_label': LABEL_NAMES[gold],
                'predicted_label': LABEL_NAMES[int(pred)],
                'correct': correct,
                'evidence': evidence,
            })
    print(f'Saved: {save_path}')


print('All functions defined.')

All functions defined.


In [25]:
# Load evidence selection model (SPICED)
EVIDENCE_MODEL_NAME = 'copenlu/spiced'
print(f'Loading evidence selection model: {EVIDENCE_MODEL_NAME}')
evidence_model = SentenceTransformer(EVIDENCE_MODEL_NAME)

# Load NLI model
NLI_MODEL_NAME = 'MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli'
print(f'Loading NLI model: {NLI_MODEL_NAME}')
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME, model_max_length=1024)
nli_model     = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME)
print('NLI model loaded.')

# Configure NCBI E-utilities
Entrez.email = os.getenv('NCBI_EMAIL', 'your@email.com')
ncbi_api_key = os.getenv('NCBI_API_KEY', '')
if ncbi_api_key:
    Entrez.api_key = ncbi_api_key
print(f'NCBI email: {Entrez.email}')
print(f'NCBI API key set: {bool(ncbi_api_key)}')
print('Setup complete.')

Loading evidence selection model: copenlu/spiced


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading NLI model: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

NLI model loaded.
NCBI email: your@email.com
NCBI API key set: False
Setup complete.


## Step 0 — Load Tweet Dataset

Loads the 60 annotated posts from `data/english/English_Tweets_Annotated.xlsx` (30 EN) and `data/spanish/Spanish_Tweets_Annotated.xlsx` (30 ES).

- **English**: pipeline input = `Tweet Text (English)` column.
- **Spanish**: pipeline input = `MT Translation (EN)` column (machine-translated to English, per the project design — the retrieval and NLI models are English-only).

Gold labels are normalised to **1 = TRUE/SUPPORTED**, **0 = FALSE/REFUTED**, **`None` = UNVERIFIABLE**. UNVERIFIABLE posts are excluded from F1/precision/recall/accuracy (no SUPPORTED/REFUTED ground truth) but are still run through the pipeline.

A heuristic check below flags any Spanish `MT Translation (EN)` cell that still looks like Spanish (i.e. wasn't actually translated) — a safeguard in case future data updates reintroduce an untranslated cell.

In [26]:
def normalize_gold_label(value):
    """Map an Excel 'Gold Label' cell to 1 (TRUE), 0 (FALSE), or None (UNVERIFIABLE/other)."""
    if value is True or value in ('True', 'TRUE', '=TRUE()'):
        return 1
    if value is False or value in ('False', 'FALSE', '=FALSE()'):
        return 0
    return None


_ES_STOPWORDS = {'el', 'la', 'los', 'las', 'que', 'de', 'en', 'un', 'una', 'es',
                 'del', 'con', 'para', 'no', 'se', 'su', 'más', 'está', 'y', 'por', 'como'}


def looks_untranslated(text):
    """Heuristic: True if `text` (after stripping URLs) still contains several common Spanish stopwords."""
    text = re.sub(r'https?://\S+', '', str(text))
    words = set(re.findall(r"[a-záéíóúñ]+", text.lower()))
    return len(words & _ES_STOPWORDS) >= 3


# English tweets
en_path = os.path.join(DATASETS_PATH, 'english', 'English_Tweets_Annotated.xlsx')
en_df = pd.read_excel(en_path, sheet_name='Annotation Tracker', header=1).iloc[:30]
en_ids    = en_df['ID'].tolist()
en_claims = en_df['Tweet Text (English)'].tolist()
en_gold   = [normalize_gold_label(v) for v in en_df['Gold Label'].tolist()]

# Spanish tweets — pipeline input is the English MT translation
es_path = os.path.join(DATASETS_PATH, 'spanish', 'Spanish_Tweets_Annotated.xlsx')
es_df = pd.read_excel(es_path, sheet_name='Annotation Tracker', header=1).iloc[:30]
es_ids    = es_df['ID'].tolist()
es_claims = es_df['MT Translation (EN)'].tolist()
es_gold   = [normalize_gold_label(v) for v in es_df['Gold Label'].tolist()]

# Flag MT translations that still look like Spanish (not actually translated)
untranslated = [tid for tid, c in zip(es_ids, es_claims) if looks_untranslated(c)]
if untranslated:
    print(f'WARNING: {len(untranslated)} "MT Translation (EN)" cells still look like Spanish '
          f'(not translated): {untranslated}')
    print('These claims will be sent to the pipeline as Spanish text — flag for Claudia to fix, '
          'or treat as a deliberate failure-mode case in the analysis.')
    print()

all_datasets = {
    'tweets_en': (en_ids, en_claims, en_gold),
    'tweets_es': (es_ids, es_claims, es_gold),
}

print('Dataset sizes:')
for name, (ids, claims, gold) in all_datasets.items():
    n_true  = sum(1 for g in gold if g == 1)
    n_false = sum(1 for g in gold if g == 0)
    n_unver = sum(1 for g in gold if g is None)
    print(f'  {name:<12}: {len(ids):>3} posts | {n_true} TRUE, {n_false} FALSE, {n_unver} UNVERIFIABLE '
          f'({n_true + n_false} scoreable)')

Dataset sizes:
  tweets_en   :  30 posts | 16 TRUE, 3 FALSE, 11 UNVERIFIABLE (19 scoreable)
  tweets_es   :  30 posts | 13 TRUE, 9 FALSE, 8 UNVERIFIABLE (22 scoreable)


## Step 1 — Document Retrieval (Wikipedia API)

For each post, `wikipedia.search()` returns the titles of the most relevant Wikipedia articles (keyword-based, similar to BM25). We then fetch the full text of each article and collect it as our evidence pool.

A 0.3-second sleep between page fetches keeps us within Wikipedia's rate limits.

In [27]:
retrieved_titles   = {}
retrieved_articles = {}

for dataset_name in DATASETS_TO_RUN:
    _, claims, _ = all_datasets[dataset_name]
    print(f'\nRetrieving Wikipedia articles for {dataset_name} ({len(claims)} claims)...')
    titles_per_claim   = []
    articles_per_claim = []

    for i, claim in enumerate(claims):
        titles, articles = retrieve_wikipedia_articles(claim)
        titles_per_claim.append(titles)
        articles_per_claim.append(articles)
        if (i + 1) % 10 == 0 or (i + 1) == len(claims):
            print(f'  {i+1}/{len(claims)} claims processed')

    retrieved_titles[dataset_name]   = titles_per_claim
    retrieved_articles[dataset_name] = articles_per_claim
    avg = np.mean([len(a) for a in articles_per_claim])
    print(f'  Done. Average articles retrieved per claim: {avg:.1f}')


Retrieving Wikipedia articles for tweets_en (30 claims)...
  10/30 claims processed
  20/30 claims processed
  30/30 claims processed
  Done. Average articles retrieved per claim: 3.2

Retrieving Wikipedia articles for tweets_es (30 claims)...
  10/30 claims processed
  20/30 claims processed
  30/30 claims processed
  Done. Average articles retrieved per claim: 0.0


In [28]:
for dataset_name in DATASETS_TO_RUN:
    ids, claims, _ = all_datasets[dataset_name]
    filepath = os.path.join(WIKI_RESULTS_PATH, f'{dataset_name}_wiki_api_titles.txt')
    with open(filepath, 'w', encoding='utf-8') as f:
        for tid, claim, titles in zip(ids, claims, retrieved_titles[dataset_name]):
            f.write(f'{tid}\t{claim}\n')
            f.write(str(titles) + '\n')
            f.write('\n')
    print(f'Saved: {filepath}')

Saved: /content/drive/MyDrive/NLP-Group-Project/pipeline/results/Wikipedia API/tweets_en_wiki_api_titles.txt
Saved: /content/drive/MyDrive/NLP-Group-Project/pipeline/results/Wikipedia API/tweets_es_wiki_api_titles.txt


## Step 2 — Evidence Selection (SPICED)

All sentences from the retrieved articles are scored against the claim using the **SPICED** sentence similarity model ([Wright et al., 2022](https://aclanthology.org/2022.emnlp-main.117.pdf)), which was shown to work well for paraphrases of scientific claims.

The top 10 sentences (by cosine similarity) are concatenated with the claim to form the input for the NLI step:

```
claim [SEP] evidence_sentence_1 evidence_sentence_2 ... evidence_sentence_10
```

In [29]:
joint_lists = {}

for dataset_name in DATASETS_TO_RUN:
    _, claims, _ = all_datasets[dataset_name]
    articles_list = retrieved_articles[dataset_name]
    print(f'\nSelecting evidence sentences for {dataset_name}...')

    joint = []
    for i, (claim, articles) in enumerate(zip(claims, articles_list)):
        evidence_str = select_top_sentences(claim, articles, evidence_model)
        joint.append(f'{claim} [SEP] {evidence_str}')
        if (i + 1) % 10 == 0 or (i + 1) == len(claims):
            print(f'  {i+1}/{len(claims)} claims processed')

    joint_lists[dataset_name] = joint
    print('  Done.')


Selecting evidence sentences for tweets_en...
  10/30 claims processed
  20/30 claims processed
  30/30 claims processed
  Done.

Selecting evidence sentences for tweets_es...
  10/30 claims processed
  20/30 claims processed
  30/30 claims processed
  Done.


In [30]:
for dataset_name in DATASETS_TO_RUN:
    filepath = os.path.join(WIKI_RESULTS_PATH, f'{dataset_name}_wiki_api_lines.txt')
    with open(filepath, 'w', encoding='utf-8') as f:
        for line in joint_lists[dataset_name]:
            f.write(line + '\n')
    print(f'Saved: {filepath}')

Saved: /content/drive/MyDrive/NLP-Group-Project/pipeline/results/Wikipedia API/tweets_en_wiki_api_lines.txt
Saved: /content/drive/MyDrive/NLP-Group-Project/pipeline/results/Wikipedia API/tweets_es_wiki_api_lines.txt


In [ ]:
# ─── RESUME POINT (Wikipedia) ────────────────────────────────────────────────
# After a session restart, run this cell (plus Drive mount, imports, load
# datasets) to reload saved Wikipedia joint lines and skip retrieval + SPICED.
# ─────────────────────────────────────────────────────────────────────────────
if 'joint_lists' not in vars() or not joint_lists:
    joint_lists = {}
    all_loaded = True
    for dataset_name in DATASETS_TO_RUN:
        filepath = os.path.join(WIKI_RESULTS_PATH, f'{dataset_name}_wiki_api_lines.txt')
        if os.path.exists(filepath):
            with open(filepath, 'r', encoding='utf-8') as f:
                joint_lists[dataset_name] = [line.rstrip('\n') for line in f]
            print(f'Loaded {dataset_name}: {len(joint_lists[dataset_name])} lines')
        else:
            print(f'No saved lines for {dataset_name} — run Wikipedia retrieval + SPICED first.')
            all_loaded = False
    if all_loaded:
        print('All Wikipedia joint lines loaded. Ready for verdict prediction.')
else:
    print('Wikipedia joint lines already in memory.')

Wikipedia joint lines already in memory.


---
## Step 3 — Verdict Prediction (DeBERTa-v3-large NLI)

The **DeBERTa-v3-large** model fine-tuned on MNLI/FEVER/ANLI predicts whether the evidence *entails* (SUPPORTED) or *contradicts* (REFUTED) the claim.

- P(entailment) > P(contradiction) → **SUPPORTED** (1)
- Otherwise → **REFUTED** (0)

A prediction is produced for **every** post. `compute_metrics()` then scores only the TRUE/FALSE subset; per-post predictions (incl. UNVERIFIABLE posts) are saved to `*_predictions.csv` for the failure taxonomy.

In [31]:
wiki_results     = {}
wiki_predictions = {}

print(f'{"Dataset":<12} {"Precision":>10} {"Recall":>10} {"F1":>10} {"Accuracy":>10} {"Scored":>10}')
print('-' * 65)

for dataset_name in DATASETS_TO_RUN:
    ids, claims, gold_labels = all_datasets[dataset_name]
    predictions = predict_verdicts(joint_lists[dataset_name], nli_model, nli_tokenizer)
    wiki_predictions[dataset_name] = predictions
    m = compute_metrics(gold_labels, predictions)
    wiki_results[dataset_name] = m
    print(f'{dataset_name:<12} {m["precision"]*100:>10.1f} {m["recall"]*100:>10.1f} '
          f'{m["f1"]*100:>10.1f} {m["accuracy"]*100:>10.1f} '
          f'{m["n_scored"]:>4}/{m["n_total"]:<4}')

print('-' * 65)
print('(precision/recall/F1/accuracy computed on TRUE/FALSE posts only; compare against')
print(' Table 3 Wikipedia BM25 / reproduction/api-pipeline Wikipedia API rows)')

Dataset       Precision     Recall         F1   Accuracy     Scored
-----------------------------------------------------------------
tweets_en         100.0       68.8       81.5       73.7   19/30  
tweets_es          62.5       76.9       69.0       59.1   22/30  
-----------------------------------------------------------------
(precision/recall/F1/accuracy computed on TRUE/FALSE posts only; compare against
 Table 3 Wikipedia BM25 / reproduction/api-pipeline Wikipedia API rows)


In [32]:
save_metrics(wiki_results, os.path.join(WIKI_RESULTS_PATH, 'metrics.csv'))

for dataset_name in DATASETS_TO_RUN:
    ids, claims, gold_labels = all_datasets[dataset_name]
    save_predictions(
        ids, claims, gold_labels,
        wiki_predictions[dataset_name], joint_lists[dataset_name],
        os.path.join(WIKI_RESULTS_PATH, f'{dataset_name}_predictions.csv'),
    )

Saved: /content/drive/MyDrive/NLP-Group-Project/pipeline/results/Wikipedia API/metrics.csv
Saved: /content/drive/MyDrive/NLP-Group-Project/pipeline/results/Wikipedia API/tweets_en_predictions.csv
Saved: /content/drive/MyDrive/NLP-Group-Project/pipeline/results/Wikipedia API/tweets_es_predictions.csv


In [ ]:
# ─── RESUME POINT (metrics + predictions) ────────────────────────────────────
# Run this cell after Drive mount + imports + load datasets to reload saved
# metrics without re-running retrieval, SPICED, or verdict prediction.
# ─────────────────────────────────────────────────────────────────────────────

if 'wiki_results' not in vars() or not wiki_results:
    _path = os.path.join(WIKI_RESULTS_PATH, 'metrics.csv')
    if os.path.exists(_path):
        wiki_results = load_metrics(_path)
        print('Loaded Wikipedia API metrics.')
        for ds, s in wiki_results.items():
            print(f'  {ds:<12}: F1={s["f1"]*100:.1f}  Acc={s["accuracy"]*100:.1f}  '
                  f'({s["n_scored"]}/{s["n_total"]} scored)')
    else:
        wiki_results = {}
        print('No saved Wikipedia API metrics — run the Wikipedia pipeline first.')
else:
    print('Wikipedia API results already in memory.')

if 'pubmed_results' not in vars() or not pubmed_results:
    _path = os.path.join(PUBMED_RESULTS_PATH, 'metrics.csv')
    if os.path.exists(_path):
        pubmed_results = load_metrics(_path)
        print('Loaded PubMed API metrics.')
        for ds, s in pubmed_results.items():
            print(f'  {ds:<12}: F1={s["f1"]*100:.1f}  Acc={s["accuracy"]*100:.1f}  '
                  f'({s["n_scored"]}/{s["n_total"]} scored)')
    else:
        pubmed_results = {}
        print('No saved PubMed API metrics — run the PubMed pipeline first.')
else:
    print('PubMed API results already in memory.')

Wikipedia API results already in memory.
No saved PubMed API metrics — run the PubMed pipeline first.


---

## PubMed API (Table 3 equivalent)

Replicates the PubMed knowledge source from the paper using the **NCBI E-utilities API** (via Biopython), the same as `reproduction/api-pipeline`.

NCBI E-utilities searches the same PubMed database using a keyword index — directly comparable to the **PubMed BM25** rows in Table 3, and to the `reproduction/api-pipeline/results/PubMed API/` baseline used in the comparison below.

Steps 2 (SPICED evidence selection) and 3 (DeBERTa NLI) are identical to the Wikipedia API section above.

> **Rate limit:** Free tier allows 3 requests/second. With a free NCBI API key the limit rises to 10 req/second. For 60 claims this section takes well under a minute either way.
>
> **Setup:** Provide any valid email address in `Entrez.email` (required by NCBI). Optionally set `Entrez.api_key` for higher throughput — both are read from `.env` in the setup cell above.

In [ ]:
retrieved_pmids           = {}
retrieved_pubmed_articles = {}

for dataset_name in DATASETS_TO_RUN:
    _, claims, _ = all_datasets[dataset_name]
    print(f'\nRetrieving PubMed abstracts for {dataset_name} ({len(claims)} claims)...')
    pmids_per_claim    = []
    articles_per_claim = []

    for i, claim in enumerate(claims):
        pmids, articles = retrieve_pubmed_articles(claim)
        pmids_per_claim.append(pmids)
        articles_per_claim.append(articles)
        if (i + 1) % 10 == 0 or (i + 1) == len(claims):
            print(f'  {i+1}/{len(claims)} claims processed')

    retrieved_pmids[dataset_name]           = pmids_per_claim
    retrieved_pubmed_articles[dataset_name] = articles_per_claim
    avg = np.mean([len(a) for a in articles_per_claim])
    print(f'  Done. Average abstracts retrieved per claim: {avg:.1f}')

# Save retrieved PMIDs — same format as *_wiki_api_titles.txt
for dataset_name in DATASETS_TO_RUN:
    ids, claims, _ = all_datasets[dataset_name]
    filepath = os.path.join(PUBMED_RESULTS_PATH, f'{dataset_name}_pubmed_api_pmids.txt')
    with open(filepath, 'w', encoding='utf-8') as f:
        for tid, claim, pmids in zip(ids, claims, retrieved_pmids[dataset_name]):
            f.write(f'{tid}\t{claim}\n')
            f.write(str(pmids) + '\n')
            f.write('\n')
    print(f'Saved: {filepath}')


Retrieving PubMed abstracts for tweets_en (30 claims)...
  10/30 claims processed
  20/30 claims processed
  30/30 claims processed
  Done. Average abstracts retrieved per claim: 1.4

Retrieving PubMed abstracts for tweets_es (30 claims)...
  10/30 claims processed
  20/30 claims processed
  30/30 claims processed
  Done. Average abstracts retrieved per claim: 0.3
Saved: /content/drive/MyDrive/NLP-Group-Project/pipeline/results/PubMed API/tweets_en_pubmed_api_pmids.txt
Saved: /content/drive/MyDrive/NLP-Group-Project/pipeline/results/PubMed API/tweets_es_pubmed_api_pmids.txt


In [ ]:
# Step 2 — SPICED evidence selection (reuses evidence_model already loaded above)
pubmed_joint_lists = {}

for dataset_name in DATASETS_TO_RUN:
    _, claims, _ = all_datasets[dataset_name]
    articles_list = retrieved_pubmed_articles[dataset_name]
    print(f'\nSelecting evidence sentences for {dataset_name}...')

    joint = []
    for i, (claim, articles) in enumerate(zip(claims, articles_list)):
        evidence_str = select_top_sentences(claim, articles, evidence_model)
        joint.append(f'{claim} [SEP] {evidence_str}')
        if (i + 1) % 10 == 0 or (i + 1) == len(claims):
            print(f'  {i+1}/{len(claims)} claims processed')

    pubmed_joint_lists[dataset_name] = joint
    print('  Done.')

# Save joint lines — same format as *_wiki_api_lines.txt
for dataset_name in DATASETS_TO_RUN:
    filepath = os.path.join(PUBMED_RESULTS_PATH, f'{dataset_name}_pubmed_api_lines.txt')
    with open(filepath, 'w', encoding='utf-8') as f:
        for line in pubmed_joint_lists[dataset_name]:
            f.write(line + '\n')
    print(f'Saved: {filepath}')


Selecting evidence sentences for tweets_en...
  10/30 claims processed
  20/30 claims processed
  30/30 claims processed
  Done.

Selecting evidence sentences for tweets_es...
  10/30 claims processed
  20/30 claims processed
  30/30 claims processed
  Done.
Saved: /content/drive/MyDrive/NLP-Group-Project/pipeline/results/PubMed API/tweets_en_pubmed_api_lines.txt
Saved: /content/drive/MyDrive/NLP-Group-Project/pipeline/results/PubMed API/tweets_es_pubmed_api_lines.txt


In [33]:
# ─── RESUME POINT (PubMed) ───────────────────────────────────────────────────
# After a session restart, run this cell (plus Drive mount, imports, load
# datasets, and the model/NCBI setup cell above) to reload saved joint lines
# and skip PubMed retrieval + SPICED evidence selection entirely.
# ─────────────────────────────────────────────────────────────────────────────
if 'pubmed_joint_lists' not in vars() or not pubmed_joint_lists:
    pubmed_joint_lists = {}
    all_loaded = True
    for dataset_name in DATASETS_TO_RUN:
        filepath = os.path.join(PUBMED_RESULTS_PATH, f'{dataset_name}_pubmed_api_lines.txt')
        if os.path.exists(filepath):
            with open(filepath, 'r', encoding='utf-8') as f:
                pubmed_joint_lists[dataset_name] = [line.rstrip('\n') for line in f]
            print(f'Loaded {dataset_name}: {len(pubmed_joint_lists[dataset_name])} lines')
        else:
            print(f'No saved lines for {dataset_name} — run PubMed retrieval + SPICED first.')
            all_loaded = False
    if all_loaded:
        print('All PubMed joint lines loaded. Ready for verdict prediction.')
else:
    print('PubMed joint lines already in memory.')

PubMed joint lines already in memory.


In [34]:
pubmed_results     = {}
pubmed_predictions = {}

print(f'{"Dataset":<12} {"Precision":>10} {"Recall":>10} {"F1":>10} {"Accuracy":>10} {"Scored":>10}')
print('-' * 65)

for dataset_name in DATASETS_TO_RUN:
    ids, claims, gold_labels = all_datasets[dataset_name]
    predictions = predict_verdicts(pubmed_joint_lists[dataset_name], nli_model, nli_tokenizer)
    pubmed_predictions[dataset_name] = predictions
    m = compute_metrics(gold_labels, predictions)
    pubmed_results[dataset_name] = m
    print(f'{dataset_name:<12} {m["precision"]*100:>10.1f} {m["recall"]*100:>10.1f} '
          f'{m["f1"]*100:>10.1f} {m["accuracy"]*100:>10.1f} '
          f'{m["n_scored"]:>4}/{m["n_total"]:<4}')

print('-' * 65)
print('(precision/recall/F1/accuracy computed on TRUE/FALSE posts only; compare against')
print(' Table 3 PubMed BM25 / reproduction/api-pipeline PubMed API rows)')

save_metrics(pubmed_results, os.path.join(PUBMED_RESULTS_PATH, 'metrics.csv'))
for dataset_name in DATASETS_TO_RUN:
    ids, claims, gold_labels = all_datasets[dataset_name]
    save_predictions(
        ids, claims, gold_labels,
        pubmed_predictions[dataset_name], pubmed_joint_lists[dataset_name],
        os.path.join(PUBMED_RESULTS_PATH, f'{dataset_name}_predictions.csv'),
    )

Dataset       Precision     Recall         F1   Accuracy     Scored
-----------------------------------------------------------------
tweets_en         100.0       87.5       93.3       89.5   19/30  
tweets_es          62.5       76.9       69.0       59.1   22/30  
-----------------------------------------------------------------
(precision/recall/F1/accuracy computed on TRUE/FALSE posts only; compare against
 Table 3 PubMed BM25 / reproduction/api-pipeline PubMed API rows)
Saved: /content/drive/MyDrive/NLP-Group-Project/pipeline/results/PubMed API/metrics.csv
Saved: /content/drive/MyDrive/NLP-Group-Project/pipeline/results/PubMed API/tweets_en_predictions.csv
Saved: /content/drive/MyDrive/NLP-Group-Project/pipeline/results/PubMed API/tweets_es_predictions.csv


---
## Comparison: Tweet Extension vs. Benchmark Baselines

Compares the extension results (English, Spanish) against two rows from Lea's `reproduction/api-pipeline/results/` (same Wikipedia API / PubMed API pipeline, unchanged):

- **SciFact** — formal scientific claims (the paper's main benchmark)
- **CoVERT** — the benchmark dataset already in tweet/social-media format (closest distributional match to our extension)

The baseline `metrics.csv` files don't include accuracy or `n_scored`/`n_total` (not computed in that notebook), so those cells are left blank for the baseline rows.

In [35]:
def load_baseline_row(results_dir, dataset):
    path = os.path.join(BASELINE_PATH, results_dir, 'metrics.csv')
    with open(path, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            if row['dataset'] == dataset:
                return row
    return None


comparison_rows = []
for source_label, results_dir, ext_results in [
    ('Wikipedia API', 'Wikipedia API', wiki_results),
    ('PubMed API',    'PubMed API',    pubmed_results),
]:
    for baseline in ['scifact', 'covert']:
        row = load_baseline_row(results_dir, baseline)
        if row:
            comparison_rows.append({
                'knowledge_source': source_label, 'dataset': baseline,
                'precision': float(row['precision']), 'recall': float(row['recall']),
                'f1': float(row['f1']), 'accuracy': '', 'n_scored': '', 'n_total': '',
            })
    for ds in DATASETS_TO_RUN:
        m = ext_results[ds]
        comparison_rows.append({
            'knowledge_source': source_label, 'dataset': ds,
            'precision': round(m['precision'] * 100, 1), 'recall': round(m['recall'] * 100, 1),
            'f1': round(m['f1'] * 100, 1), 'accuracy': round(m['accuracy'] * 100, 1),
            'n_scored': m['n_scored'], 'n_total': m['n_total'],
        })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

comparison_path = os.path.join(COMPARISON_PATH, 'extension_vs_baseline.csv')
comparison_df.to_csv(comparison_path, index=False)
print(f'Saved: {comparison_path}')

,knowledge_source,dataset,precision,recall,f1,accuracy,n_scored,n_total
0,Wikipedia API,scifact,73.5,83.3,78.1,,,
1,Wikipedia API,covert,79.2,59.6,68.0,,,
2,Wikipedia API,tweets_en,100.0,68.8,81.5,73.7,19,30
3,Wikipedia API,tweets_es,62.5,76.9,69.0,59.1,22,30
4,PubMed API,scifact,75.3,77.6,76.5,,,
5,PubMed API,covert,79.1,63.1,70.2,,,
6,PubMed API,tweets_en,100.0,87.5,93.3,89.5,19,30
7,PubMed API,tweets_es,62.5,76.9,69.0,59.1,22,30


Saved: /content/drive/MyDrive/NLP-Group-Project/pipeline/results/comparison/extension_vs_baseline.csv


---
## Failure Taxonomy

This cell pulls together two kinds of cases for qualitative review (deliverable: 5–10 documented failure cases with hypotheses):

1. **Scoring failures** — TRUE/FALSE posts where the predicted verdict disagrees with the gold label.
2. **Forced verdicts on UNVERIFIABLE posts** — the NLI model always outputs SUPPORTED or REFUTED, even for posts the annotators marked UNVERIFIABLE. These aren't "errors" against a gold label, but they show how the pipeline behaves on claims it has no business verifying — likely a rich source of failure-mode examples for noisy/informal text.

For each case, the claim text and the retrieved evidence (the `[SEP]`-joined sentences actually fed to DeBERTa) are shown so you can form a hypothesis about *why* it failed (e.g. irrelevant article retrieved, SPICED picked an unrelated sentence, claim too informal/ambiguous for NLI, MT translation issue for Spanish, etc.).

In [36]:
def build_failure_table(results_path_map):
    rows = []
    for source_label, results_path in results_path_map.items():
        for dataset_name in DATASETS_TO_RUN:
            pred_path = os.path.join(results_path, f'{dataset_name}_predictions.csv')
            with open(pred_path, encoding='utf-8') as f:
                for row in csv.DictReader(f):
                    if row['correct'] == 'N' or row['gold_label'] == 'UNVERIFIABLE':
                        rows.append({
                            'knowledge_source': source_label,
                            'dataset': dataset_name,
                            'id': row['id'],
                            'claim': row['claim'],
                            'gold_label': row['gold_label'],
                            'predicted_label': row['predicted_label'],
                            'evidence': row['evidence'][:300],
                        })
    return pd.DataFrame(rows)


failure_df = build_failure_table({
    'Wikipedia API': WIKI_RESULTS_PATH,
    'PubMed API':    PUBMED_RESULTS_PATH,
})

print(f'{len(failure_df)} candidate failure cases (scoring errors + forced UNVERIFIABLE verdicts)')
failure_path = os.path.join(COMPARISON_PATH, 'failure_candidates.csv')
failure_df.to_csv(failure_path, index=False)
print(f'Saved: {failure_path}')
display(failure_df)

63 candidate failure cases (scoring errors + forced UNVERIFIABLE verdicts)
Saved: /content/drive/MyDrive/NLP-Group-Project/pipeline/results/comparison/failure_candidates.csv


,knowledge_source,dataset,id,claim,gold_label,predicted_label,evidence
0,Wikipedia API,tweets_en,EN-01,"Astronomers found a planet that has no land, n...",UNVERIFIABLE,SUPPORTED,
1,Wikipedia API,tweets_en,EN-05,BREAKING NEWS: 'Negative time' confirmed: Mind...,UNVERIFIABLE,SUPPORTED,
2,Wikipedia API,tweets_en,EN-06,A vaccine that could provide protection agains...,SUPPORTED,REFUTED,
3,Wikipedia API,tweets_en,EN-08,he use of antidepressants while pregnant does ...,SUPPORTED,REFUTED,
4,Wikipedia API,tweets_en,EN-09,Revolution's pancreatic cancer drug doubles su...,SUPPORTED,REFUTED,31 may – a phase 3 trial published in the new ...
...,...,...,...,...,...,...,...
58,PubMed API,tweets_es,ES-24,Turmeric is as effective as ibuprofen in reduc...,SUPPORTED,REFUTED,
59,PubMed API,tweets_es,ES-25,I got vaccinated last week and I did have side...,UNVERIFIABLE,REFUTED,
60,PubMed API,tweets_es,ES-26,Your testosterone starts to decline at 35. By ...,UNVERIFIABLE,REFUTED,
61,PubMed API,tweets_es,ES-27,Patients in rooms with southeast-facing window...,UNVERIFIABLE,SUPPORTED,


### Taxonomy categories (fill in after reviewing `failure_candidates.csv`)

Pick 5–10 representative cases from the table above and categorise them. Suggested starting categories — adjust based on what you actually see:

| Category | Description | Example IDs |
|---|---|---|
| Irrelevant retrieval | Retrieved Wikipedia/PubMed article doesn't match the claim's topic | |
| Evidence-selection mismatch | Right article retrieved, but SPICED picked unrelated sentences | |
| Informal-language NLI failure | Evidence is relevant, but DeBERTa fails on informal/sarcastic phrasing | |
| Forced verdict on UNVERIFIABLE | Pipeline confidently verdicts a claim annotators couldn't verify | |
| MT translation issue (Spanish only) | Mistranslation (or untranslated text — see ES-02/ES-26/ES-29) changes the claim's meaning before retrieval | |
| Numeric/statistical precision | Claim hinges on a specific number the evidence doesn't address | |

Write up each category with 1–2 concrete examples (claim, evidence snippet, predicted vs. gold, your hypothesis) in `documentation/technical_report.md` under "Extension Results" / "Failure Analysis".